# 1. Introduction

This notebook provides an exploratory data analysis (EDA) for the **Maze Crawler Episodes** ecosystem. Maze Crawler replays are aggregated daily and published as separate Kaggle datasets. 

In this notebook, we look at the high-level trends using the `manifest.csv` index, then unpack the structure of the simulation replays across consecutive days, analyze step-by-step observations, and explore reward distributions. This top-down approach gives us a strong baseline understanding of how episode stats form over the course of time and what winning strategies might look like.

In [ ]:
import json
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Professional, clean visual styling
sns.set_theme(style="whitegrid", rc={"axes.spines.right": False, "axes.spines.top": False})
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

# 2. Daily Trends from the Index Manifest

Multiple daily datasets are tracked inside the `maze-crawler-episodes-index` dataset. We'll start by analyzing the `manifest.csv` to see how the episode submission counts, sizes, and scores evolve each day.

In [ ]:
INDEX_DIR = '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-index'
# Fallback to local files if developing locally
if not os.path.exists(INDEX_DIR):
    INDEX_DIR = '.' # Adjust as needed locally

# Note: local path for testing is set based on the user's attachment
manifest_path = os.path.join(INDEX_DIR, 'manifest.csv')
if not os.path.exists(manifest_path):
    manifest_path = r'c:\Users\danus\Downloads\manifest.csv'

if os.path.exists(manifest_path):
    df_manifest = pd.read_csv(manifest_path)
    df_manifest['date'] = pd.to_datetime(df_manifest['date'])
    display(df_manifest.head())

    # Plot Episode Counts and Total Bytes over time
    fig, ax1 = plt.subplots()
    
    color1 = '#4A5FBF'
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Episode Count', color=color1)
    ax1.plot(df_manifest['date'], df_manifest['episode_count'], color=color1, marker='o', linewidth=2)
    ax1.tick_params(axis='y', labelcolor=color1)
    
    ax2 = ax1.twinx()
    color2 = '#C75B3F'
    ax2.set_ylabel('Total Bytes (GB)', color=color2)
    ax2.plot(df_manifest['date'], df_manifest['total_bytes'] / (1024**3), color=color2, marker='s', linewidth=2)
    ax2.tick_params(axis='y', labelcolor=color2)
    
    plt.title('Daily Episode Submissions and Storage Volume')
    fig.tight_layout()
    plt.show()

    # Plot Scores over time
    plt.figure()
    plt.plot(df_manifest['date'], df_manifest['top_avg_score'], label='Top Avg Score', marker='o', linewidth=2)
    plt.plot(df_manifest['date'], df_manifest['median_avg_score'], label='Median Avg Score', marker='s', linewidth=2)
    plt.xlabel('Date')
    plt.ylabel('Score')
    plt.title('Daily Episode Scores Evolution')
    plt.legend()
    plt.show()
else:
    print("manifest.csv not found.")

# 3. Reading Simulation JSON Replays

Now we'll load a sample of the actual match replays. These daily directories contain many large JSON files representing individual games. We'll pick one to unpack the simulation dynamics and configuration.

In [ ]:
# Depending on your environment, you'd iterate the multiple daily paths
# Here we'll check paths programmatically
daily_paths = [
    '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-2026-05-01',
    '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-2026-05-02',
    '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-2026-05-03',
    '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-2026-05-04',
    '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-2026-05-05',
    '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-2026-05-06',
    '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-2026-05-07',
    '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-2026-05-08',
    '/kaggle/input/datasets/organizations/kaggle/maze-crawler-episodes-2026-05-09'
]

# For local testing, we have a sample in the current working directory
sample_files = glob.glob('*.json') 
for dp in daily_paths:
    if os.path.exists(dp):
        sample_files.extend(glob.glob(os.path.join(dp, '*.json')))

if sample_files:
    sample_path = sample_files[0]
    with open(sample_path, 'r') as f:
        episode_data = json.load(f)
    print(f"Loaded sample episode: {os.path.basename(sample_path)}")
    
    print("\n--- Match Outcome ---")
    statuses = episode_data.get('statuses', [])
    rewards = episode_data.get('rewards', [])
    for agent_id, (status, reward) in enumerate(zip(statuses, rewards)):
        print(f"Agent {agent_id} -> Status: {status}, Final Reward: {reward}")

    print("\n--- Episode Configuration (subset) ---")
    config = episode_data.get('configuration', {})
    for k in ['episodeSteps', 'height', 'crystalDensity', 'miningNodeDensity', 'energyPerTurn']:
        if k in config:
            print(f"{k}: {config[k]}")
else:
    print("No episode JSON files found. Are the Kaggle datasets attached?")

# 4. Step-by-Step Telemetry Analysis

Every episode consists of a `steps` array, detailing each turn of the simulation. Each element in `steps` is a list holding the states for each player (Agent 0 and Agent 1).
Let's analyze the progression of rewards (scores) and the actions taken over the course of the episode.

In [ ]:
if sample_files:
    steps = episode_data.get('steps', [])
    print(f"Total steps in this episode: {len(steps)}")
    
    agent_0_rewards = []
    agent_1_rewards = []
    agent_0_actions = []
    agent_1_actions = []

    for step_idx, step_data in enumerate(steps):
        # step_data is a list with entries for each agent
        if len(step_data) > 0:
            agent_0 = step_data[0]
            agent_0_rewards.append(agent_0.get('reward', 0) if agent_0.get('reward') is not None else 0)
            
            # Actions are represented as dictionaries mapping entity IDs to action strings
            a0_action_dict = agent_0.get('action') or {}
            agent_0_actions.extend(a0_action_dict.values())
            
        if len(step_data) > 1:
            agent_1 = step_data[1]
            agent_1_rewards.append(agent_1.get('reward', 0) if agent_1.get('reward') is not None else 0)
            
            a1_action_dict = agent_1.get('action') or {}
            agent_1_actions.extend(a1_action_dict.values())

    # Plot Reward Progression
    plt.figure()
    plt.plot(range(len(agent_0_rewards)), agent_0_rewards, label='Agent 0 Reward', color='#4A5FBF', linewidth=2)
    plt.plot(range(len(agent_1_rewards)), agent_1_rewards, label='Agent 1 Reward', color='#C75B3F', linewidth=2)
    plt.xlabel('Step')
    plt.ylabel('Accumulated Reward')
    plt.title('Reward Progression over Episode Steps')
    plt.legend()
    plt.show()

### 4.1 Action Distribution

Let's look at the frequency of different actions commanded by the agents during the episode. This gives us insight into how active the agents were and what movement/mining patterns they favored.

In [ ]:
if sample_files:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # We will use pandas value_counts to easily plot distributions
    if agent_0_actions:
        pd.Series(agent_0_actions).value_counts().plot(kind='bar', ax=axes[0], color='#4A5FBF')
        axes[0].set_title('Agent 0 Actions')
        axes[0].set_ylabel('Count')
        axes[0].tick_params(axis='x', rotation=45)
    else:
        axes[0].text(0.5, 0.5, "No Actions", ha='center', va='center')
        axes[0].set_title('Agent 0 Actions')
        
    if agent_1_actions:
        pd.Series(agent_1_actions).value_counts().plot(kind='bar', ax=axes[1], color='#C75B3F')
        axes[1].set_title('Agent 1 Actions')
        axes[1].set_ylabel('Count')
        axes[1].tick_params(axis='x', rotation=45)
    else:
        axes[1].text(0.5, 0.5, "No Actions", ha='center', va='center')
        axes[1].set_title('Agent 1 Actions')

    plt.tight_layout()
    plt.show()

# 5. Advanced Analysis: Multi-Episode Aggregates & Strategy Dynamics

While analyzing a single episode gives us a good sense of action distributions, looking across multiple episodes provides deeper insights into winning margins and match durations. We'll iterate over a larger subset of available json replays to plot reward distributions globally.

In [ ]:
# Let's parse up to 100 sample episodes to aggregate match stats
MAX_EPISODES_TO_PARSE = 100
aggregated_stats = []

valid_files = [f for f in sample_files if os.path.exists(f)][:MAX_EPISODES_TO_PARSE]

for fp in valid_files:
    try:
        with open(fp, 'r') as f:
            ep = json.load(f)
            
        rewards = ep.get('rewards', [0, 0])
        steps = ep.get('steps', [])
        
        # In Maze Crawler arrays, Agent 0 and Agent 1 can both score
        if len(rewards) >= 2:
            score_diff = abs(rewards[0] - rewards[1])
            winner_score = max(rewards)
            loser_score = min(rewards)
        else:
            score_diff, winner_score, loser_score = 0, 0, 0
            
        aggregated_stats.append({
            'file': os.path.basename(fp),
            'steps': len(steps),
            'agent_0_reward': rewards[0] if len(rewards) > 0 else 0,
            'agent_1_reward': rewards[1] if len(rewards) > 1 else 0,
            'score_diff': score_diff,
            'winner_score': winner_score,
            'loser_score': loser_score
        })
    except Exception as e:
        continue

df_stats = pd.DataFrame(aggregated_stats)

if not df_stats.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Scatter: Steps vs absolute score difference
    sns.scatterplot(data=df_stats, x='steps', y='score_diff', alpha=0.7, color='#4A5FBF', s=80, ax=axes[0])
    axes[0].set_title('Match Duration vs Margin of Victory')
    axes[0].set_xlabel('Total Match Steps')
    axes[0].set_ylabel('Absolute Score Difference')

    # KDE Plot for winning vs losing scores
    sns.kdeplot(df_stats['winner_score'], fill=True, label='Winner Score', color='#3D6E5C', ax=axes[1])
    sns.kdeplot(df_stats['loser_score'], fill=True, label='Loser Score', color='#C75B3F', ax=axes[1])
    axes[1].set_title('Score Density: Winners vs Losers')
    axes[1].set_xlabel('Final Reward')
    axes[1].set_ylabel('Density')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Not enough episodes loaded to aggregate.")

### 5.1 Momentum Tracking

Do teams snowball early or do comebacks happen? To check this, we look at the reward accumulation velocity computed over a rolling window. This highlights the exploration vs. exploitation phase transition.

In [ ]:
if len(agent_0_rewards) > 10: # Ensure we have enough steps mapped from section 4
    # Compute rolling differentials in rewards (momentum)
    window_size = max(5, int(len(agent_0_rewards) * 0.1)) # Dynamic window size to prevent NaNs in shorter games
    a0_momentum = pd.Series(agent_0_rewards).diff().rolling(window=window_size).mean()
    a1_momentum = pd.Series(agent_1_rewards).diff().rolling(window=window_size).mean()

    plt.figure(figsize=(10, 5))
    plt.plot(a0_momentum, label=f'Agent 0 Score Momentum (MA={window_size})', color='#4A5FBF', linewidth=2)
    plt.plot(a1_momentum, label=f'Agent 1 Score Momentum (MA={window_size})', color='#C75B3F', linewidth=2)
    plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
    plt.xlabel('Step')
    plt.ylabel('Average Points Gained Per Step')
    plt.title('Momentum Swings: Finding the Exploitation Phase')
    plt.legend()
    plt.show()
else:
    print(f"Skipping Momentum plot: The match only had {len(agent_0_rewards)} steps, too short to compute momentum rolling averages.")

### 5.2 Environmental Evolution (Mining Nodes & Crystals)

To capture the physical map dynamics, we can extract counts from the `observation` dictionary passed to Agent 0 at each step. This allows us to track the discovery/creation of items like crystals, nodes, and robots over the lifetime of the match.

In [ ]:
if sample_files:
    crystals_count = []
    robots_count = []
    mining_nodes_count = []

    for step_data in steps:
        if len(step_data) > 0:
            obs = step_data[0].get('observation', {})
            # Counting the items parsed in the observation schema
            crystals_count.append(len(obs.get('globalCrystals', {})))
            robots_count.append(len(obs.get('robots', {})))
            mining_nodes_count.append(len(obs.get('globalMiningNodes', {})))

    fig, ax1 = plt.subplots(figsize=(10, 6))

    color_crystal = '#3D6E5C'
    ax1.plot(crystals_count, label='Crystals (Global)', color=color_crystal, linewidth=2)
    ax1.plot(mining_nodes_count, label='Mining Nodes (Global)', color='#C75B3F', linestyle='--', linewidth=2)
    
    ax1.set_xlabel('Match Step')
    ax1.set_ylabel('Item Count')
    
    # Using a secondary axis for robot counts, as they scale differently
    ax2 = ax1.twinx()
    color_robot = '#4A4A6A'
    ax2.plot(robots_count, label='Active Robots', color=color_robot, linewidth=2)
    ax2.set_ylabel('Active Robots', color=color_robot)
    ax2.tick_params(axis='y', labelcolor=color_robot)

    # Combined legends
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

    plt.title('Evolution of Map Entities Over Time')
    plt.show()

### 5.3 Initial Board State Visualization

Lastly, we can visualize the generated starting layout for our sample episode. Extracting point coordinates for crystals, mining nodes, and robots directly maps the topological starting point for the agents relative to resource density.

In [ ]:
if sample_files and len(steps) > 0:
    obs = steps[0][0].get('observation', {})
    
    # Extract positions (keys are formatted as "x,y")
    def parse_coords(d):
        coords = []
        for k in d.keys():
            try:
                coords.append(tuple(map(int, k.split(','))))
            except ValueError:
                pass
        return coords
    
    crystals = parse_coords(obs.get('globalCrystals', {}))
    nodes = parse_coords(obs.get('globalMiningNodes', {}))
    
    # Robots format: {'id': [player, x, y, energy, ...]}
    robot_coords = []
    robot_colors = []
    for r_id, r_data in obs.get('robots', {}).items():
        if len(r_data) >= 3:
            robot_coords.append((r_data[1], r_data[2])) 
            robot_colors.append('#4A5FBF' if str(r_data[0]) == '0' else '#C75B3F')
            
    fig, ax = plt.subplots(figsize=(8, 8))
    
    if crystals:
        cx, cy = zip(*crystals)
        ax.scatter(cx, cy, label='Crystals', color='#7AAE99', marker='d', s=60, alpha=0.8)
        
    if nodes:
        nx, ny = zip(*nodes)
        ax.scatter(nx, ny, label='Mining Nodes', color='#A14E3A', marker='s', s=100)
        
    if robot_coords:
        rx, ry = zip(*robot_coords)
        ax.scatter(rx, ry, label='Robots', c=robot_colors, marker='o', s=200, edgecolor='white', linewidth=2)
        
    ax.set_title('Initial Maze Topology and Resource Distribution')
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate (North Scrolling)')
    
    # Invert Y if visually treating 0,0 at top-left
    ax.invert_yaxis()
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1))
    ax.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

# 6. Conclusion

This notebook established a quick baseline and exploratory analysis onto the `maze-crawler-episodes-index` meta-data as well as unpacking step-by-step game JSON files across multiple consecutive days. 

By analyzing the advanced momentum dynamics and score aggregation, we demonstrated how agent leads develop and the typical volume output across multiple datasets. From here, further steps for modeling could involve extracting feature representations of the `observation` dictionary from each step (e.g. mapping walls, crystals, and mines), and training policies for an offline reinforcement learning agent or imitation learning model based on top players.